# 03 - Functions & OOP for ML/SWE Roles

Part of the Python, DSA & Git chapter. This notebook covers the Python that shows up in real production ML and backend code: advanced function signatures, type hints, decorators you will actually write, and the OOP patterns behind frameworks like PyTorch and scikit-learn.

Covers: *args/**kwargs, positional/keyword-only params, type hints, closures, functools, decorators from scratch, classes, magic methods, classmethod/staticmethod/property, inheritance vs composition, abstract base classes, dataclasses, and a worked mini-project. Practice exercises at the end.

# Part A - Functions, Leveled Up

## *args and **kwargs

*args collects extra positional arguments into a tuple; **kwargs collects extra keyword arguments into a dict. These show up constantly in decorators and wrapper functions (later in this notebook) because they let a function forward an arbitrary call signature to another function.

In [1]:
def describe(*args, **kwargs):
    print("positional args:", args)
    print("keyword args:", kwargs)

describe(1, 2, 3, name="alice", age=30)

def call_with_logging(fn, *args, **kwargs):
    print(f"calling {fn.__name__} with args={args}, kwargs={kwargs}")
    return fn(*args, **kwargs)

def add(a, b):
    return a + b

result = call_with_logging(add, 3, 4)
print("result:", result)

# Unpacking works the other way too -- spreading a list/dict into a call
nums = [3, 4]
print(add(*nums))

opts = {"a": 10, "b": 20}
print(add(**opts))

positional args: (1, 2, 3)
keyword args: {'name': 'alice', 'age': 30}
calling add with args=(3, 4), kwargs={}
result: 7
7
30


## Positional-only and keyword-only parameters

Since Python 3.8, a / in a function signature marks everything before it as positional-only (cannot be passed by keyword); a * marks everything after it as keyword-only (must be passed by keyword). This is the kind of signature seen constantly in library code, and it is worth being able to read and write.

In [2]:
def connect(host, port, /, *, timeout=30, retries=3):
    return f"connecting to {host}:{port} (timeout={timeout}, retries={retries})"

print(connect("localhost", 8080))                      # host, port positional -- fine
print(connect("localhost", 8080, timeout=5))            # timeout must be keyword

try:
    connect(host="localhost", port=8080)                # host/port are positional-only
except TypeError as e:
    print("positional-only violated:", e)

try:
    connect("localhost", 8080, 5)                       # timeout is keyword-only
except TypeError as e:
    print("keyword-only violated:", e)

connecting to localhost:8080 (timeout=30, retries=3)
connecting to localhost:8080 (timeout=5, retries=3)
positional-only violated: connect() got some positional-only arguments passed as keyword arguments: 'host, port'
keyword-only violated: connect() takes 2 positional arguments but 3 were given


## Type hints

Type hints document, and let tools check, what a function expects and returns. Python does NOT enforce them at runtime by default -- they exist for readability, IDE autocomplete, and static analysis tools like mypy, and they are what libraries like FastAPI and Pydantic use to validate data automatically (notebook 07).

In [3]:
from typing import List, Dict, Optional, Union, Callable

def average(nums: List[float]) -> float:
    return sum(nums) / len(nums)

print(average([1.0, 2.0, 3.0]))

# Modern Python (3.9+) allows built-in generics directly, no typing import needed
def average_modern(nums: list[float]) -> float:
    return sum(nums) / len(nums)

# Optional[X] means Union[X, None] -- "this might be X, or might be missing"
def find_user(user_id: int, cache: Optional[Dict[int, str]] = None) -> Optional[str]:
    if cache is None:
        cache = {}
    return cache.get(user_id)

print(find_user(1))
print(find_user(1, {1: "alice"}))

# Union -- accepts more than one type
def to_seconds(value: Union[int, float, str]) -> float:
    return float(value)

print(to_seconds("3.5"), to_seconds(2))

# Callable -- typing a function as a parameter, used constantly for callbacks
def apply_twice(fn: Callable[[int], int], x: int) -> int:
    return fn(fn(x))

print(apply_twice(lambda x: x * 2, 3))

# Type hints are NOT enforced at runtime -- Python runs this without complaint
def double(x: int) -> int:
    return x * 2

print(double(5))
print(double("ab"), "-- hint said int, Python ran it anyway since it is only documentation")

2.0
None
alice
3.5 2.0
12
10
abab -- hint said int, Python ran it anyway since it is only documentation


## Closures and first-class functions

Functions are ordinary objects in Python -- they can be assigned to variables, passed as arguments, returned from other functions, and stored in data structures. A closure is a nested function that remembers variables from its enclosing scope even after that scope has finished executing. This mechanism is exactly what makes decorators possible.

In [4]:
def make_multiplier(factor):
    def multiplier(x):
        return x * factor          # `factor` is captured from the enclosing scope
    return multiplier

double_fn = make_multiplier(2)
triple_fn = make_multiplier(3)
print(double_fn(5), triple_fn(5))   # each closure remembers its OWN factor

# Functions as values: store them in a dict, pick one at runtime
operations = {
    "add": lambda a, b: a + b,
    "sub": lambda a, b: a - b,
    "mul": lambda a, b: a * b,
}
print(operations["mul"](4, 5))

# A classic closure gotcha: late binding in loops
funcs = []
for i in range(3):
    funcs.append(lambda: i)          # all three closures capture the SAME variable i
print([f() for f in funcs])          # -> [2, 2, 2], NOT [0, 1, 2] -- a very common interview trap

funcs_fixed = []
for i in range(3):
    funcs_fixed.append(lambda i=i: i)   # default-argument trick forces early binding
print([f() for f in funcs_fixed])       # -> [0, 1, 2]

10 15
20
[2, 2, 2]
[0, 1, 2]


## functools: partial, reduce, lru_cache

functools has three tools worth knowing cold: partial (pre-fill some arguments of a function), reduce (fold a sequence down to a single value), and lru_cache (automatic memoization, caching return values by argument -- big for expensive repeated calls like feature lookups or API calls).

In [5]:
from functools import partial, reduce, lru_cache

def power(base, exponent):
    return base ** exponent

square = partial(power, exponent=2)     # pre-fill exponent -- square is now a 1-arg function
cube = partial(power, exponent=3)
print(square(5), cube(5))

nums = [1, 2, 3, 4, 5]
total = reduce(lambda acc, x: acc + x, nums)              # fold: ((((1+2)+3)+4)+5)
product = reduce(lambda acc, x: acc * x, nums, 1)          # with an explicit start value
print(total, product)

call_count = 0

@lru_cache(maxsize=None)
def slow_fib(n):
    global call_count
    call_count += 1
    if n < 2:
        return n
    return slow_fib(n - 1) + slow_fib(n - 2)

print(slow_fib(30))
print("actual function calls, thanks to caching:", call_count)   # far fewer than 2^30

25 125
15 120
832040
actual function calls, thanks to caching: 31


## Decorators, from first principles

A decorator is a function that takes a function and returns a, usually wrapped, function. The @decorator syntax is sugar for func = decorator(func). Understanding this mechanically, not just as "the @ thing," is what makes it possible to write custom decorators instead of only using library ones.

In [6]:
def shout(fn):
    def wrapper(*args, **kwargs):
        result = fn(*args, **kwargs)
        return result.upper() if isinstance(result, str) else result
    return wrapper

@shout
def greet(name):
    return f"hello, {name}"

print(greet("ada"))          # equivalent to: greet = shout(greet)

# Without functools.wraps, the wrapped function loses its original identity --
# a common source of confusing bugs (broken docstrings, broken introspection).
print("name without functools.wraps:", greet.__name__)   # -> "wrapper", not "greet" -- bad!

from functools import wraps

def shout_fixed(fn):
    @wraps(fn)                     # preserves __name__, __doc__, etc of the original function
    def wrapper(*args, **kwargs):
        result = fn(*args, **kwargs)
        return result.upper() if isinstance(result, str) else result
    return wrapper

@shout_fixed
def greet2(name):
    '''Return a friendly greeting.'''
    return f"hello, {name}"

print(greet2("ada"), "| name:", greet2.__name__, "| doc:", greet2.__doc__)

HELLO, ADA
name without functools.wraps: wrapper
HELLO, ADA | name: greet2 | doc: Return a friendly greeting.


## A decorator worth having memorized: @timer

Timing how long a function takes is one of the most common reasons to write a custom decorator, and it demonstrates why *args/**kwargs matter here: the wrapper needs to work for ANY function signature, not just one specific one.

In [7]:
import time
from functools import wraps

def timer(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = fn(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{fn.__name__} took {elapsed*1000:.3f} ms")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

_ = slow_sum(5_000_000)

slow_sum took 61.209 ms


## A decorator worth having memorized: @retry

Flaky network and API calls are a fact of life in production ML systems: calling an external model API, a feature store, a database. Note that retry(max_attempts=5, delay=0) is a **decorator factory** -- a function that returns a decorator -- which is why it needs an extra layer of nesting compared to @timer above: retry(...) runs first and returns the actual decorator, which then gets applied to the function.

In [8]:
def retry(max_attempts=3, delay=0.0):
    def decorator(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            last_exception = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return fn(*args, **kwargs)
                except Exception as e:
                    last_exception = e
                    print(f"  attempt {attempt} failed: {e}")
                    time.sleep(delay)
            raise last_exception
        return wrapper
    return decorator

flaky_call_count = 0

@retry(max_attempts=5, delay=0)
def unreliable_api_call():
    global flaky_call_count
    flaky_call_count += 1
    if flaky_call_count < 3:
        raise ConnectionError("simulated network failure")
    return "success!"

print(unreliable_api_call())

  attempt 1 failed: simulated network failure
  attempt 2 failed: simulated network failure
success!


## Reimplementing memoization from scratch

Seeing how lru_cache works under the hood removes the mystery -- it is a decorator holding a dict, nothing more exotic than that.

In [9]:
def memoize(fn):
    cache = {}
    @wraps(fn)
    def wrapper(*args):
        if args not in cache:
            cache[args] = fn(*args)
        return cache[args]
    wrapper.cache = cache             # expose the cache for inspection, handy for debugging
    return wrapper

@memoize
def slow_square(x):
    time.sleep(0.05)          # simulate an expensive computation
    return x * x

start = time.perf_counter()
slow_square(5)
slow_square(5)                # served from cache -- should be near-instant the second time
elapsed = time.perf_counter() - start
print(f"two calls with memoization: {elapsed*1000:.1f} ms (should be ~50ms, not ~100ms)")
print("cache contents:", slow_square.cache)

two calls with memoization: 53.2 ms (should be ~50ms, not ~100ms)
cache contents: {(5,): 25}


# Part B - OOP the Way it Shows Up in ML Code

## Instance vs class attributes

A class attribute is shared across every instance; an instance attribute belongs to just one object. This distinction causes a very common bug: mutable class attributes such as a list get shared across every instance unless created inside __init__ instead.

In [10]:
class Model:
    framework = "scikit-learn"          # CLASS attribute -- shared by every instance

    def __init__(self, name):
        self.name = name                # INSTANCE attribute -- unique per object
        self.is_trained = False

m1 = Model("logistic_regression")
m2 = Model("random_forest")
print(m1.name, m2.name)
print(m1.framework, m2.framework)

Model.framework = "pytorch"             # changing the CLASS attribute affects ALL instances
print(m1.framework, m2.framework)

class WorseLogger:
    logs = []                # BAD if this is meant to be per-instance -- it is shared!

a, b = WorseLogger(), WorseLogger()
a.logs.append("a message")
print("b also sees a message because logs is a SHARED class attribute:", b.logs)

logistic_regression random_forest
scikit-learn scikit-learn
pytorch pytorch
b also sees a message because logs is a SHARED class attribute: ['a message']


## Magic methods: __repr__ and __str__

__repr__ is the unambiguous, developer-facing representation, ideally something that could be pasted back into Python to recreate the object. __str__ is the human-readable, user-facing one. If only one is defined, define __repr__ -- Python falls back to it for print() when __str__ is missing.

In [11]:
class Experiment:
    def __init__(self, name, accuracy):
        self.name = name
        self.accuracy = accuracy

    def __repr__(self):
        return f"Experiment(name={self.name!r}, accuracy={self.accuracy})"

    def __str__(self):
        return f"{self.name}: {self.accuracy:.1%} accuracy"

e = Experiment("baseline_v1", 0.873)
print(e)          # uses __str__
print(repr(e))    # uses __repr__
print([e])        # containers always use __repr__ of their elements, even when printed

baseline_v1: 87.3% accuracy
Experiment(name='baseline_v1', accuracy=0.873)
[Experiment(name='baseline_v1', accuracy=0.873)]


## Magic methods: __len__ and __getitem__

These two are what make an object work with len(obj), obj[i], and iteration via for x in obj. This is not incidental trivia -- it is literally the interface PyTorch expects for its Dataset class: any class implementing __len__ and __getitem__ can be handed to a DataLoader.

In [12]:
class SimpleDataset:
    '''A minimal stand-in for torch.utils.data.Dataset.'''
    def __init__(self, features, labels):
        assert len(features) == len(labels)
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

ds = SimpleDataset(features=[[1, 2], [3, 4], [5, 6]], labels=[0, 1, 0])
print(len(ds))               # calls __len__
print(ds[1])                  # calls __getitem__
for x, y in ds:                # __getitem__ + __len__ together are enough to make this iterable
    print(x, "->", y)

3
([3, 4], 1)
[1, 2] -> 0
[3, 4] -> 1
[5, 6] -> 0


## Magic methods: __call__

Implementing __call__ lets an object be invoked like a function: obj(x) instead of obj.some_method(x). This is why in PyTorch the pattern is model(x) rather than model.forward(x) -- nn.Module.__call__ runs some bookkeeping such as hooks, then calls forward internally.

In [13]:
class LinearModel:
    def __init__(self, weight, bias):
        self.weight = weight
        self.bias = bias

    def __call__(self, x):
        return self.weight * x + self.bias

model = LinearModel(weight=2.0, bias=1.0)
print(model(3))          # looks like a function call, but model is an OBJECT
print(model(10))
print(callable(model))    # True -- objects with __call__ pass the callable() check

7.0
21.0
True


## @classmethod, @staticmethod, and @property

- @staticmethod: a function living in the class namespace that does not touch the instance or class at all -- just a namespacing convenience.
- @classmethod: receives the class itself (cls) instead of an instance -- the standard way to write alternate constructors.
- @property: makes a method callable like an attribute, obj.x instead of obj.x() -- allows validation or computation behind what looks like a plain attribute, without changing calling code.

In [14]:
class Temperature:
    def __init__(self, celsius):
        self._celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("temperature below absolute zero")
        self._celsius = value

    @property
    def fahrenheit(self):                     # a READ-ONLY computed property
        return self._celsius * 9 / 5 + 32

    @classmethod
    def from_fahrenheit(cls, f):
        return cls((f - 32) * 5 / 9)           # alternate constructor

    @staticmethod
    def is_valid(celsius):
        return celsius >= -273.15

t = Temperature(25)
print(t.celsius, t.fahrenheit)        # accessed like plain attributes, no parentheses
t.celsius = 30                        # goes through the setter validation
print(t.celsius)

try:
    t.celsius = -300
except ValueError as e:
    print("validation caught:", e)

t2 = Temperature.from_fahrenheit(98.6)
print(f"{t2.celsius:.1f}C")

print(Temperature.is_valid(-500))     # called on the CLASS, no instance needed

25

 77.0
30
validation caught: temperature below absolute zero
37.0C
False


## Inheritance vs composition

Inheritance models an is-a relationship; composition models a has-a relationship. A useful default: prefer composition unless polymorphism is genuinely needed, meaning treating different subclasses interchangeably through a shared interface -- deep inheritance hierarchies get brittle fast.

In [15]:
# Inheritance: RandomForestModel IS-A BaseModel
class BaseModel:
    def __init__(self, name):
        self.name = name

    def summary(self):
        return f"Model: {self.name}"

class RandomForestModel(BaseModel):
    def __init__(self, name, n_trees):
        super().__init__(name)             # call the parent __init__
        self.n_trees = n_trees

    def summary(self):
        base = super().summary()            # extend, do not just replace, the parent behavior
        return f"{base} ({self.n_trees} trees)"

rf = RandomForestModel("forest_v1", n_trees=100)
print(rf.summary())

# Composition: a Pipeline HAS-A list of preprocessing steps and a model -- not an is-a anything
class Pipeline:
    def __init__(self, steps, model):
        self.steps = steps        # each step is a callable: data -> transformed data
        self.model = model

    def run(self, data):
        for step in self.steps:
            data = step(data)
        return self.model(data)

normalize = lambda x: x / 100
add_bias = lambda x: x + 1

pipeline = Pipeline(steps=[normalize, add_bias], model=LinearModel(weight=2.0, bias=0.0))
print(pipeline.run(50))

Model: forest_v1 (100 trees)


3.0


## Abstract base classes

An ABC defines an interface that subclasses are forced to implement -- attempting to instantiate a subclass that skips a required method raises a TypeError immediately, rather than failing mysteriously later when that method finally gets called. This is exactly the pattern behind scikit-learn fit/predict and PyTorch forward.

In [16]:
from abc import ABC, abstractmethod

class Estimator(ABC):
    @abstractmethod
    def fit(self, X, y):
        ...

    @abstractmethod
    def predict(self, X):
        ...

    def fit_predict(self, X, y):            # concrete method, free to every subclass
        self.fit(X, y)
        return self.predict(X)

class MeanBaseline(Estimator):
    def fit(self, X, y):
        self.mean_ = sum(y) / len(y)
        return self

    def predict(self, X):
        return [self.mean_] * len(X)

model = MeanBaseline()
print(model.fit_predict(X=[1, 2, 3], y=[10, 20, 30]))

try:
    class Incomplete(Estimator):
        def fit(self, X, y):          # forgot to implement predict
            return self
    Incomplete()
except TypeError as e:
    print("ABC enforcement caught it:", e)

[20.0, 20.0, 20.0]


ABC enforcement caught it: Can't instantiate abstract class Incomplete without an implementation for abstract method 'predict'


## @dataclass for config objects

Passing around a sprawling pile of **kwargs or a raw dict for configuration is error-prone -- typos in keys fail silently, there is no autocomplete, no type checking. @dataclass generates __init__, __repr__, and __eq__ automatically from type-annotated fields, making it the standard modern choice for config objects such as training hyperparameters.

In [17]:
from dataclasses import dataclass, field

@dataclass
class TrainingConfig:
    learning_rate: float = 1e-3
    batch_size: int = 32
    epochs: int = 10
    optimizer: str = "adam"
    layer_sizes: list = field(default_factory=lambda: [128, 64])   # mutable default needs field()

config = TrainingConfig(learning_rate=5e-4, epochs=20)
print(config)                                    # auto-generated __repr__
print(config.batch_size)                         # plain attribute access
print(config == TrainingConfig(learning_rate=5e-4, epochs=20))   # auto-generated __eq__

# Why field(default_factory=...) instead of a plain mutable default:
try:
    @dataclass
    class Bad:
        items: list = []             # dataclass explicitly rejects this at class-definition time
except ValueError as e:
    print("dataclass caught the mutable-default trap:", e)

TrainingConfig(learning_rate=0.0005, batch_size=32, epochs=20, optimizer='adam', layer_sizes=[128, 64])
32
True
dataclass caught the mutable-default trap: mutable default <class 'list'> for field items is not allowed: use default_factory


# Part C - Mini-Project: a PyTorch-Shaped Dataset + Model

Putting __len__, __getitem__, __call__, inheritance, and ABCs together in one place -- deliberately shaped like real PyTorch and scikit-learn code, so the magic methods above stop being trivia and start being the reason that framework code looks the way it does.

In [18]:
class Dataset(ABC):
    @abstractmethod
    def __len__(self):
        ...

    @abstractmethod
    def __getitem__(self, idx):
        ...

class TabularDataset(Dataset):
    def __init__(self, rows, labels):
        self.rows = rows
        self.labels = labels

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx], self.labels[idx]

class Module(ABC):
    @abstractmethod
    def forward(self, x):
        ...

    def __call__(self, x):              # this IS the __call__ -> forward indirection PyTorch uses
        return self.forward(x)

class TinyLinearModel(Module):
    def __init__(self, weight, bias):
        self.weight = weight
        self.bias = bias

    def forward(self, x):
        return [self.weight * xi + self.bias for xi in x]

def train_loop(dataset, model, epochs=3):
    for epoch in range(epochs):
        total_error = 0
        for i in range(len(dataset)):              # uses __len__
            x, y_true = dataset[i]                  # uses __getitem__
            y_pred = model(x)                       # uses __call__ -> forward
            error = sum(abs(p - t) for p, t in zip(y_pred, [y_true] * len(y_pred)))
            total_error += error
        print(f"epoch {epoch}: total_error={total_error:.2f}")

ds = TabularDataset(rows=[[1, 2], [3, 4], [5, 6]], labels=[3, 7, 11])
model = TinyLinearModel(weight=2.0, bias=1.0)
train_loop(ds, model)

print()
print("Note: this is deliberately not doing real gradient-based training -- it exists to show the")
print("__len__ / __getitem__ / __call__ SHAPE real frameworks use, not to be a working optimizer.")

epoch 0: total_error=6.00
epoch 1: total_error=6.00
epoch 2: total_error=6.00

Note: this is deliberately not doing real gradient-based training -- it exists to show the
__len__ / __getitem__ / __call__ SHAPE real frameworks use, not to be a working optimizer.


## Practice exercises

Implement each TODO, then run the check cell. These specifically exercise decorators and magic methods.

In [19]:
def count_calls(fn):
    # Decorator: wrap fn so that wrapper.calls tracks how many times it has been called.
    # TODO: implement using a closure variable and functools.wraps
    raise NotImplementedError

class Vector2D:
    # A 2D vector supporting +, -, ==, and a readable repr.
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):
        # TODO: return a new Vector2D that is the component-wise sum
        raise NotImplementedError

    def __sub__(self, other):
        # TODO: return a new Vector2D that is the component-wise difference
        raise NotImplementedError

    def __eq__(self, other):
        # TODO: return True if x and y both match
        raise NotImplementedError

    def __repr__(self):
        # TODO: return something like "Vector2D(x=1, y=2)"
        raise NotImplementedError

class Stack:
    # A stack (LIFO) wrapping a list, exposing push/pop/peek and __len__/__bool__.
    def __init__(self):
        self._items = []

    def push(self, item):
        # TODO: implement
        raise NotImplementedError

    def pop(self):
        # TODO: implement -- remove and return the top item
        raise NotImplementedError

    def peek(self):
        # TODO: implement -- return the top item WITHOUT removing it
        raise NotImplementedError

    def __len__(self):
        # TODO: implement
        raise NotImplementedError

    def __bool__(self):
        # TODO: implement -- True if and only if the stack is non-empty
        raise NotImplementedError

In [20]:
def _check(label, ok, detail=""):
    print("  [" + ("PASS" if ok else "FAIL") + "]", label, detail)

# --- count_calls ---
try:
    @count_calls
    def add_one(x):
        return x + 1
    add_one(1); add_one(2); add_one(3)
    _check("count_calls tracks call count", add_one.calls == 3, f"(calls={getattr(add_one, 'calls', '?')})")
except NotImplementedError:
    print("  [SKIP] count_calls -- not implemented yet")
except Exception as e:
    print("  [ERROR] count_calls --", e)

# --- Vector2D ---
try:
    v1, v2 = Vector2D(1, 2), Vector2D(3, 4)
    _check("Vector2D.__add__", (v1 + v2) == Vector2D(4, 6))
    _check("Vector2D.__sub__", (v2 - v1) == Vector2D(2, 2))
    _check("Vector2D.__eq__", Vector2D(1, 2) == Vector2D(1, 2))
    _check("Vector2D.__repr__ is informative", "1" in repr(v1) and "2" in repr(v1), repr(v1))
except NotImplementedError:
    print("  [SKIP] Vector2D -- not implemented yet")
except Exception as e:
    print("  [ERROR] Vector2D --", e)

# --- Stack ---
try:
    s = Stack()
    _check("empty stack is falsy", not bool(s))
    s.push(1); s.push(2); s.push(3)
    _check("len after 3 pushes", len(s) == 3)
    _check("peek does not remove", s.peek() == 3 and len(s) == 3)
    _check("pop returns LIFO order", [s.pop(), s.pop(), s.pop()] == [3, 2, 1])
    _check("empty again after popping all", len(s) == 0 and not bool(s))
except NotImplementedError:
    print("  [SKIP] Stack -- not implemented yet")
except Exception as e:
    print("  [ERROR] Stack --", e)

  [SKIP] count_calls -- not implemented yet
  [SKIP] Vector2D -- not implemented yet
  [SKIP] Stack -- not implemented yet


## Self-check before moving on

- [ ] I can write a function using *args/**kwargs and explain when to reach for each
- [ ] I can read and write positional-only (/) and keyword-only (*) parameter signatures
- [ ] I know type hints are not enforced at runtime, and what tools actually use them
- [ ] I can explain closures and the late-binding-in-loops gotcha
- [ ] I can write a decorator from scratch, including one that takes its own arguments (a decorator factory)
- [ ] I know why functools.wraps matters
- [ ] I can explain __len__/__getitem__/__call__ and connect them directly to the PyTorch Dataset and nn.Module APIs
- [ ] I know when to reach for @classmethod vs @staticmethod vs @property
- [ ] I can explain why ABCs enforce an interface at instantiation time, not just by convention
- [ ] I default to @dataclass over raw dicts or **kwargs for config objects

Next: `04-iterators-generators-context-managers.ipynb`